In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from category_encoders.target_encoder import TargetEncoder
from xgboost import XGBClassifier, XGBRegressor
from skopt.space import Real, Categorical, Integer
from skopt import BayesSearchCV
from sklearn.metrics import mean_squared_error, r2_score
#pip install xgboost
#pip install --upgrade category_encoders
#pip install scikit-optimize

df = pd.read_csv('bank-additional-full.csv', delimiter=';')
#getting rid of features that are not very useful
cols_to_drop = ['duration', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'poutcome', 'default']
#rename columns so they are undersetandable
df.head()


df = df.drop(columns=cols_to_drop).rename(columns={'job': 'job_type', 
                                                   'housing': 'housing_loan_status', 'loan': 'personal_loan_status', 
                                                   'contact': 'contact_type', 'month': 'contact_month', 
                                                   'day_of_week': 'contact_day_of_week', 'campaign': 'num_contacts', 
                                                   'pdays': 'days_last_contact', 'previous': 'previous_contacts', 
                                                   'y': 'result'
                                                    })
# convert the target to numerical values
df['result'] = df['result'].replace({'yes': 1, 'no': 0})
df['result'] = pd.to_numeric(df['result'], errors='coerce')

# 2. Drop any rows where 'result' is now NaN (missing)


# 3. Force the column to be a pure Integer (not float)
df['result'] = df['result'].astype(int)
df['result'].value_counts()


#Can use XGBRegressor instead for regression models
cat_cols = ['job_type', 'marital', 'education', 'housing_loan_status', 'personal_loan_status', 'contact_type', 'contact_month', 'contact_day_of_week']
#te = TargetEncoder(cat_cols)
#df[cat_cols] = te.fit_transform(df[cat_cols], df['result'])

df['result'].value_counts()


X = df.drop(columns='result')
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')


params = {
    'objective': 'reg:squarederror',
    'max_depth': 3,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'enable_categorical': True
}
model = XGBRegressor(**params)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")
print(f"Dtype of y_train: {y_train.dtype}")
print(f"Unique values in y_train: {y_train.unique()}")


Mean Squared Error: 0.08
R-squared: 0.20
Dtype of y_train: int64
Unique values in y_train: [0 1]
